# Main Comparison / Bakeoff (M3) — Colab Launcher

The headline empirical chapter: baseline vs COSGD vs BoGrad vs dropout vs
GradDrop, 4 base optimizers x 6 datasets, **tuned on val, reported on test**,
multi-seed with paired data order. Results -> Drive; idempotent (resume).

**This is large** (6 x 4 x 5 x 5 = 600 tuned runs + tuning). Run **per-dataset**
across sessions — everything resumes from `results.json` + cached `best.json`.

## 1. Clone / pull + deps + Drive

In [ ]:
import os, subprocess, sys, importlib, pathlib, shutil
REPO_URL="https://github.com/rayden96/MastersDissertationExperiments.git"
BRANCH="m0-infrastructure"; REPO_DIR="/content/MastersDissertationExperiments"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git","clone","--branch",BRANCH,REPO_URL,REPO_DIR],check=True)
else:
    subprocess.run(["git","-C",REPO_DIR,"pull","origin",BRANCH],check=True)
for pkg,pn in [("sklearn","scikit-learn"),("datasets","datasets")]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.run([sys.executable,"-m","pip","install","-q",pn],check=True)
from google.colab import drive; drive.mount("/content/drive")
RESULTS_ROOT="/content/drive/MyDrive/dissertation/results"; os.makedirs(RESULTS_ROOT,exist_ok=True)
os.environ["DISSERTATION_RESULTS_ROOT"]=RESULTS_ROOT
# persist the bakeoff _core + _sensitivity + _scale + _gradstats dirs on Drive
mc = pathlib.Path(REPO_DIR)/"PaperReadyExperiments"/"30_main_comparison"
for sub in ("_core","_sensitivity","_scale","_gradstats"):
    local=mc/sub; drive_dir=pathlib.Path(RESULTS_ROOT)/"30_main_comparison"/sub
    drive_dir.mkdir(parents=True,exist_ok=True)
    if local.exists() and not local.is_symlink():
        for it in local.glob('*'): shutil.move(str(it),str(drive_dir/it.name))
        shutil.rmtree(local,ignore_errors=True)
    if not local.exists(): os.symlink(drive_dir,local,target_is_directory=True)
import torch; print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. Smoke (~3 min CPU/GPU)

In [ ]:
%cd {REPO_DIR}/PaperReadyExperiments/30_main_comparison
!python run.py --smoke && python views.py

## 3. Bakeoff — ONE DATASET AT A TIME
Each dataset is a self-contained chunk. Do the cheap ones first; re-run any cell
to resume. (Drop `--no-measure` if you want the interference metrics logged too —
slower.)

In [ ]:
%cd {REPO_DIR}/PaperReadyExperiments/30_main_comparison
!python run.py --datasets mnist           --seeds 2026 2027 2028 2029 2030
!python run.py --datasets cifar10         --seeds 2026 2027 2028 2029 2030

In [ ]:
%cd {REPO_DIR}/PaperReadyExperiments/30_main_comparison
!python run.py --datasets emnist_balanced --seeds 2026 2027 2028 2029 2030
!python run.py --datasets covertype       --seeds 2026 2027 2028 2029 2030
!python run.py --datasets yahoo_answers    --seeds 2026 2027 2028 2029 2030

In [ ]:
%cd {REPO_DIR}/PaperReadyExperiments/30_main_comparison
# CIFAR-100 + ResNet is the heaviest; COSGD at 100 classes is slow (expected).
!python run.py --datasets cifar100 --seeds 2026 2027 2028

## 4. Sensitivity / scale / grad-stats axes (30.04-30.08)

In [ ]:
%cd {REPO_DIR}/PaperReadyExperiments/30_main_comparison
!python sensitivity.py --kind lr    --datasets cifar10 cifar100
!python sensitivity.py --kind K     --datasets cifar10 mnist emnist_balanced
!python sensitivity.py --kind batch --datasets cifar10 emnist_balanced
!python scale_gradstats.py --kind scale
!python scale_gradstats.py --kind gradstats

## 5. Build all figures + tables from the canonical records

In [ ]:
%cd {REPO_DIR}/PaperReadyExperiments/30_main_comparison
!python views.py